# 第2回 宿題：設定やデータを変えて LoRA 学習を繰り返す（Colab 用）

演習のノートブック（`notebooks/lora_format_tuning.ipynb`）のセクション 1〜7 と同じ処理を `scripts/lora_homework.py` にまとめ、1 回の実行（ラン）ごとに別のプロセスで動かします。
プロセスを分けるので、「設定を変えたら ランタイム → セッションを再起動 して上から実行する」のと同じ条件になります。

| ラン | 用途 | 変えるもの |
| --- | --- | --- |
| `baseline` | 演習・基礎の基準 | なし（演習のノートブックの `CONFIG` のまま） |
| `epochs5` | 基礎 | `num_train_epochs` 20→5 |
| `crypto` | 実装 | FAQ を架空の暗号資産取引所に置き換える（`scripts/lora_data.py`） |
| `glossary` | 発展 | 用語集から作った SFT データ（`lectures/02/data/glossary_sft.jsonl`）で学習し、用語集の検索と比べる |
| `lr5e-5`（任意） | 基礎 | `lr` 2e-4→5e-5 |
| `r4`（任意） | 基礎 | `lora_r` 16→4 |

**進め方**

1. **ファイル → ドライブにコピーを保存** し、**ランタイム → ランタイムのタイプを変更** で T4 GPU になっていることを確かめる。
2. 上から順に実行する（**ランタイム → すべてのセルを実行** でもよい）。セクション 1 で Google Drive へのアクセスを 1 回だけ許可する。
3. 時間の目安：必須のラン（セクション 4）は合わせて 45〜50 分、任意のラン（セクション 5）は 15 分ほど。`glossary` の時間は T4 で測っていないので、目安より長くなることがある。実行中はタブを開いたままにしておく。
4. 結果は Drive の `aie2026/lec02/<ラン>/` に保存される。途中で切断されても、もう一度上から実行すれば、終わったランは飛ばして続きから実行する。
5. 終わったら **ランタイム → ランタイムを接続解除して削除**。

**出ても問題ないもの**：pip の赤い `ERROR: pip's dependency resolver ...`、`You are sending unauthenticated requests to the HF Hub`、`... is falling back to its reference PyTorch implementation ...`、`The tokenizer has new PAD/BOS/EOS tokens ...`（演習のノートブックと同じ）。

## 0. GPU の確認とライブラリの導入（演習のノートブックと同じ）

1 つ目のセルに `Tesla T4` と表示されることを確かめます。2 つ目のセルは導入に 1 分ほどかかります。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 結果の保存先（Google Drive）

結果（`result.json`・`eval_*.json`・`run.log`）を Drive の `aie2026/lec02/<ラン>/` に保存します。アクセスの許可を求められたら許可してください。
許可しなかった場合は Colab の中（`/content/out`）に保存します。この場合はセッションが切れると消えるので、最後のセクション 6 で zip をダウンロードしてください。

In [ ]:
from pathlib import Path

OUT = Path("/content/drive/MyDrive/aie2026/lec02")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:   # 許可しなかった場合などは、Colab の中だけに保存する（セッションが切れると消える）
    print("Drive をマウントできませんでした:", e)
    OUT = Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)
print("結果の保存先:", OUT)

## 2. 宿題のコードを取得する（fork の `lec02` ブランチ）

実行するたびに取り直すので、いつも最新のコードで動きます。表示されたコミットは `result.json` の `env.commit` にも記録されます。

In [ ]:
REPO_URL = "https://github.com/Aimitmk/lecture-ai-engineering-2026.git"
BRANCH = "lec02"
REPO = Path("/content/lecture-ai-engineering-2026")
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --format="%h %s"

## 3. ランを実行する関数

1 つのランを `python -m scripts.lora_homework --run <ラン>` として別のプロセスで実行し、出力をここに流しながら `run.log` にも保存します。
保存先に `result.json` があるランは、終わったものとして飛ばします（やり直すときは、そのランのフォルダを Drive から消してください）。
チェックポイントと adapter は `/content/lec02_work/` に置き、Drive には保存しません。

In [ ]:
import os, subprocess, sys, time

def run(name):
    out = OUT / name
    if (out / "result.json").exists():
        print(f"[{name}] 実行済みなので飛ばします（やり直すときは {out} を消してください）")
        return
    out.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-u", "-m", "scripts.lora_homework", "--run", name, "--out", str(out), "--work", f"/content/lec02_work/{name}"]
    env = {**os.environ, "TQDM_MININTERVAL": "20"}   # 進捗バーの表示を 20 秒ごとに減らす（学習には影響しない）
    t0 = time.time()
    with open(out / "run.log", "w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        proc.wait()
    print(f"[{name}] 終了コード {proc.returncode} | {time.time() - t0:.0f} 秒")
    if proc.returncode != 0:
        raise RuntimeError(f"{name} が失敗しました。{out / 'run.log'} の最後を確認してください")

## 4. 必須のラン（合わせて 45〜50 分）

`baseline`（演習・基礎の基準）→ `epochs5`（基礎）→ `crypto`（実装）→ `glossary`（発展）の順に実行します。
最初のランだけ、モデルのダウンロード（約 1.75GB）に 1 分ほど余分にかかります。

ランごとに、次のものが表示されます。

- 学習前・学習後の要約：`format_rate`・`content_rate` など。`glossary` はカテゴリごとの正解率。
- `trainable params`、loss の記録（10 ステップごと）、`学習時間: ... 秒 | ステップ数: ...`。
- 学習後に測った valid loss（`train:` の行）。

In [ ]:
for name in ["baseline", "epochs5", "crypto", "glossary"]:
    run(name)

## 5. 任意のラン（合わせて 15 分ほど）

基礎の比較を増やすランです。時間がなければ飛ばしてかまいません。後から実行するときも、セクション 0〜3 を上から実行してからこのセルを実行してください。

In [ ]:
for name in ["lr5e-5", "r4"]:
    run(name)

## 6. 集計とダウンロード

保存したすべてのランの `result.json` を、表にまとめて表示します。
Drive に保存していない場合や、手元に結果を残したい場合は、2 つ目のセルで zip をダウンロードしてください。

In [ ]:
from IPython.display import Markdown, display

table = subprocess.run([sys.executable, "-m", "scripts.lora_homework", "--summary", "--results", str(OUT)], cwd=REPO, capture_output=True, text=True)
display(Markdown(table.stdout or table.stderr))

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/lec02_results", "zip", OUT)
files.download(archive)